In [1]:
# config file paths
import os

EVENTS_CSV_PATH = os.path.join(
    "x-rays E = 1.0keV for filtering",
    "filtered_before_erass2_overlap_regions_and_observation_time_>=0.1_scattering_probability_and_<=0.950212931632136_absorption_probability.csv",
)
BOUNDARY_DIR = os.path.join(
    "x-rays E = 1.0keV for filtering",
    "overlap_boundaries",
)

print("EVENTS_CSV_PATH exists:", os.path.exists(EVENTS_CSV_PATH))
print("BOUNDARY_DIR exists:", os.path.exists(BOUNDARY_DIR))

EVENTS_CSV_PATH exists: True
BOUNDARY_DIR exists: True


In [2]:
# Boundary polygon -> covering eROSITA skytiles

"""
The GW boundary polygons span tens of degrees. Lay down a grid of sample points across the
polygon's bounding box, query the skytile API at each point, and
collect the unique set of skytiles that come back.

"""

import csv
import requests
import numpy as np


def load_boundary_polygon(boundary_json_path):
    blobs = {}
    with open(boundary_json_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            blob_id = int(row["blob_id"])
            ra = float(row["ra"])
            dec = float(row["dec"])
            blobs.setdefault(blob_id, []).append((ra, dec))
    return blobs


def point_in_polygon(ra, dec, polygon_ra, polygon_dec):
    # ra, dec is some point
    # polygon_ra, polygon_dec are points on the boundaries
    n = len(polygon_ra) # number of points
    inside = False
    j = n - 1 # last index
    for i in range(n):
        # get next edge
        xi, yi = polygon_ra[i], polygon_dec[i] # vertex 1
        xj, yj = polygon_ra[j], polygon_dec[j] #vertex 2
        if ((yi > dec) != (yj > dec)) and (
            ra < (xj - xi) * (dec - yi) / (yj - yi + 1e-15) + xi
        ): 
            # point must be vertically (dec) wihtin the vertices, 
            # (xj - xi) * (dec - yi) / (yj - yi + 1e-15) + xi is the ra coord where edge hits dec
            # ra < ... tests if point is to the left of the edge
            inside = not inside # counts crossing from left to right. Odd number means point is inside
        j = i # test every edge
    return inside


def sample_grid_inside_polygon(polygon_ra, polygon_dec, step_deg=1.5):
    """
    Lay a regular grid over the polygon's bounding box at `step_deg`
    spacing, keep only the points (grid intersections) that fall inside the polygon. 
    This is to figure out which skytiles we should use.

    step_deg should be smaller than the skytile size (3.6 deg) so we
    don't skip over a tile - 1.5 deg gives good coverage with margin.
    """
    ra_arr = np.array(polygon_ra)
    dec_arr = np.array(polygon_dec)

    ra_grid = np.arange(ra_arr.min(), ra_arr.max() + step_deg, step_deg)
    dec_grid = np.arange(dec_arr.min(), dec_arr.max() + step_deg, step_deg)

    points = []
    for ra in ra_grid:
        for dec in dec_grid:
            if point_in_polygon(ra, dec, polygon_ra, polygon_dec):
                points.append((ra, dec))
    return points

def find_skytiles_at_point(ra_icrs, dec_icrs, search_radius_deg=0.1):
    """
    Calls the eROSITA skytile search API for a single sky position.

    IMPORTANT: ra_icrs/dec_icrs must be equatorial (ICRS) coordinates
    in decimal degrees - the API's RA/DEC parameters are documented
    with a plain equatorial example (RA=180&DEC=-45) and there is no
    coordinate-system parameter, unlike the *manual* skytile search
    web form (which has an ICRS/FK5/Galactic/Ecliptic/FK4 dropdown -
    that dropdown applies to the web UI only, not this API).

    Our boundary JSON files provide both ra/dec (equatorial) and l/b
    (galactic) per vertex - load_boundary_polygon() already reads the
    ra/dec columns, so values flowing into this function are correct
    as long as you don't substitute l/b in by mistake upstream.
    """
    url = (
        "https://erosita.mpe.mpg.de/dr1/erodat/skyview/skytile_search_api/"
        f"?RA={ra_icrs}&DEC={dec_icrs}&RAD={search_radius_deg}"
    )
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return resp.json().get("tiles", [])


def find_all_skytiles_for_blob(polygon_ra, polygon_dec, step_deg=1.5):
    """
    Full procedure: grid-sample the polygon, query each sample point,
    de-duplicate by srvmap tile number.
    """
    sample_points = sample_grid_inside_polygon(polygon_ra, polygon_dec, step_deg)
    seen_tiles = {}  # survey map -> tile info dict

    for ra, dec in sample_points:
        tiles = find_skytiles_at_point(ra, dec)
        for t in tiles:
            seen_tiles[t["srvmap"]] = t # t["srvmap"] gets the unique 6-digit eROSITA Tile ID

    return list(seen_tiles.values()), sample_points


def find_all_skytiles_for_event(boundary_json_path, step_deg=1.5):
    """
    Like find_all_skytiles_for_blob, but covers every blob in the
    boundary file (the main patch plus any small disjoint fragments),
    de-duplicated across all of them. This is what you want for "one
    search per event" - the search region is the union of every blob.
    """
    blobs = load_boundary_polygon(boundary_json_path)
    seen_tiles = {}
    all_samples = []

    for blob_id, points in blobs.items():
        ra_list = [p[0] for p in points]
        dec_list = [p[1] for p in points]

        if len(points) < 3:
            # Degenerate blob (1-2 points) - can't form a polygon.
            # Treat the point(s) themselves as the sample(s).
            samples = points
        else:
            samples = sample_grid_inside_polygon(ra_list, dec_list, step_deg)
            if not samples:
                # blob is smaller than step_deg in both directions -
                # grid sampling found nothing inside; fall back to
                # using its vertices directly so it's not dropped.
                samples = points

        all_samples.extend(samples)
        for ra, dec in samples:
            tiles = find_skytiles_at_point(ra, dec)
            for t in tiles:
                seen_tiles[t["srvmap"]] = t

    return list(seen_tiles.values()), all_samples, blobs


In [ ]:
# %% [markdown]
# ## Step 2: theta(dt) radius window, crop, edge-detect, circular Hough transform

# %%
"""
Step 2/3: given a skytile srvmap number, download its FITS image,
mask to the GW boundary polygon's footprint, and run a circular
Hough transform whose search radius is derived per-observation from
the dust-echo physics: theta proportional to sqrt(delta_t).

Image scale (from eROSITA DR1 docs): 3240x3240 px covering 3.6x3.6 deg
  -> 4 arcsec/pixel

Dust-echo radius-time relation:
  delta_t ~ theta^2 / 2   (the proportionality constant folds in dust
  distance and geometry, which we don't know exactly per-source - so
  we calibrate it from the one data point we were given: theta=30
  arcmin at delta_t=365 days, i.e. theta(dt) = 30 * sqrt(dt_days/365)).

Because that calibration point itself has real uncertainty (unknown
dust-sheet distance, departure from the small-angle approximation,
etc.), we don't search a single radius - we search a band around the
predicted theta, controlled by RADIUS_UNCERTAINTY_FRAC below.
"""

import io
import re
import requests
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS

ARCSEC_PER_PIXEL = 4.0

# Calibration point you were given: 1 year -> 30 arcmin
REF_DT_DAYS = 365.0
REF_THETA_ARCMIN = 30.0

# How much slack to put around the predicted theta, since the
# proportionality constant isn't known precisely. +-20% is a
# reasonable starting point; tighten it if you trust the calibration
# more, widen it if candidates are coming up empty.
RADIUS_UNCERTAINTY_FRAC = 0.20


def parse_relative_time_to_days(time_str):
    """
    Parses the CSV's 'time_relative_to_gw_detection' format, e.g.
    '+122d 11h 26m 1.9s' -> 122.476... (float days)
    """
    m = re.match(r"\+?(\d+)d\s+(\d+)h\s+(\d+)m\s+([\d.]+)s", time_str)
    if not m:
        raise ValueError(f"Unrecognized time format: {time_str!r}")
    d, h, mi, se = m.groups()
    return float(d) + float(h) / 24 + float(mi) / 1440 + float(se) / 86400


def predicted_theta_arcmin(dt_days):
    """theta(dt) = REF_THETA_ARCMIN * sqrt(dt_days / REF_DT_DAYS)"""
    return REF_THETA_ARCMIN * np.sqrt(dt_days / REF_DT_DAYS)


def predicted_radius_window_arcmin(dt_days, uncertainty_frac=RADIUS_UNCERTAINTY_FRAC):
    """Returns (theta_min, theta_max, theta_center) in arcmin for this dt."""
    theta = predicted_theta_arcmin(dt_days)
    return theta * (1 - uncertainty_frac), theta * (1 + uncertainty_frac), theta


def arcmin_to_pixels(arcmin):
    return (arcmin * 60.0) / ARCSEC_PER_PIXEL


def build_download_url(srvmap, product="EXP", proc="010", band=4):
    """
    Archive layout and filename pattern confirmed against a real
    directory listing (srvmap 174079):
      directory: /{RRR}/{DDD}/{PRODUCT}_{proc}/   (srvmap = RRRDDD)
      filename:  em01_{DDD}{RRR}_0{band}{filter}_Image_c{proc}.fits.gz
                 (note: filename's tile-number field is DDD+RRR -
                 REVERSED from the directory's RRR/DDD order)
    band=4 -> 0.2-2.3 keV, eROSITA's standard combined soft+medium
    band. Valid Image bands seen in a real EXP_010 listing: 1-7
    (021..027); there's no band-0 Image (only EventList).
    """
    srvmap_str = f"{int(srvmap):06d}"
    rrr, ddd = srvmap_str[:3], srvmap_str[3:]
    directory = f"https://erosita.mpe.mpg.de/dr1/erodat/data/download/{rrr}/{ddd}/{product}_{proc}/"
    filename = f"em01_{ddd}{rrr}_02{band}_Image_c{proc}.fits.gz"
    return directory + filename


def download_fits(url):
    """
    Downloads and opens a FITS file. Transparently handles gzip
    (.fits.gz) - astropy.io.fits.open auto-detects gzip from the
    byte stream regardless of the URL's extension, so no special
    handling is needed beyond keeping the raw bytes intact.
    """
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    return fits.open(io.BytesIO(resp.content))


def crop_to_polygon(image_data, wcs, polygon_ra, polygon_dec):
    """
    Build a boolean mask the same shape as image_data, True only for
    pixels whose sky coordinate falls inside the polygon. Then zero
    out everything outside it.

    Uses astropy's WCS to convert pixel grid -> ra/dec, then the same
    point-in-polygon test as step 1.
    """
    h, w = image_data.shape
    yy, xx = np.mgrid[0:h, 0:w]
    ra, dec = wcs.wcs_pix2world(xx, yy, 0)

    mask = np.zeros((h, w), dtype=bool)
    n = len(polygon_ra)

    # Vectorized point-in-polygon (ray casting) over the whole pixel grid
    inside = np.zeros((h, w), dtype=bool)
    j = n - 1
    for i in range(n):
        xi, yi = polygon_ra[i], polygon_dec[i]
        xj, yj = polygon_ra[j], polygon_dec[j]
        cond = (yi > dec) != (yj > dec)
        # avoid div-by-zero where yj == yi
        denom = (yj - yi)
        denom = np.where(denom == 0, 1e-15, denom)
        x_intersect = (xj - xi) * (dec - yi) / denom + xi
        toggle = cond & (ra < x_intersect)
        inside = np.where(toggle, ~inside, inside)
        j = i
    mask = inside

    cropped = np.where(mask, image_data, 0)
    return cropped, mask


def simple_edge_map(image, threshold_sigma=2.0, ignore_mask=None):
    """Same gradient-based edge detector as before, mask-aware."""
    gy, gx = np.gradient(image.astype(float))
    grad_mag = np.sqrt(gx**2 + gy**2)

    valid = grad_mag[ignore_mask] if ignore_mask is not None else grad_mag
    med = np.median(valid)
    mad = np.median(np.abs(valid - med))
    sigma_est = 1.4826 * mad

    edge_mask = grad_mag > (med + threshold_sigma * sigma_est)
    if ignore_mask is not None:
        edge_mask &= ignore_mask
    return edge_mask


def circular_hough_transform(edge_mask, radii, vote_threshold_frac=0.5, n_angles=360):
    """Same voting algorithm as before, radii now in pixel units."""
    H, W = edge_mask.shape
    radii = list(radii)
    accumulator = np.zeros((len(radii), H, W), dtype=np.int32)

    ys, xs = np.nonzero(edge_mask)
    thetas = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    cos_t = np.cos(thetas)
    sin_t = np.sin(thetas)

    for r_idx, r in enumerate(radii):
        a_offsets = (r * cos_t).astype(np.int32)
        b_offsets = (r * sin_t).astype(np.int32)

        for x, y in zip(xs, ys):
            a_candidates = x - a_offsets
            b_candidates = y - b_offsets
            valid = (
                (a_candidates >= 0) & (a_candidates < W) &
                (b_candidates >= 0) & (b_candidates < H)
            )
            accumulator[r_idx, b_candidates[valid], a_candidates[valid]] += 1

    candidates = []
    for r_idx, r in enumerate(radii):
        layer = accumulator[r_idx]
        if layer.max() == 0:
            continue
        cutoff = vote_threshold_frac * layer.max()
        by, bx = np.nonzero(layer >= cutoff)
        for y, x in zip(by, bx):
            candidates.append((y, x, r, int(layer[y, x])))
    return accumulator, candidates


def run_halo_search(image_data, wcs, polygon_ra, polygon_dec, dt_days,
                     radius_step_px=1, vote_threshold_frac=0.6,
                     min_mask_pixels=100):
    """
    End-to-end: crop to the GW polygon, edge-detect, Hough-transform
    over the theta(dt)-predicted radius band only, return ranked
    candidates with sky coordinates.

    dt_days: time since GW detection for *this* observation row
             (parse with parse_relative_time_to_days on the CSV's
             time_relative_to_gw_detection column).

    min_mask_pixels: if the polygon doesn't overlap this tile at all
             (or only barely - e.g. a sliver of the boundary clips a
             tile's corner), the crop mask will be empty or nearly
             so. There's nothing to search in that case, so return
             an empty result immediately rather than running edge
             detection on (effectively) no data, which previously
             produced NaN/median-of-empty-slice warnings downstream.
    """
    theta_min, theta_max, theta_center = predicted_radius_window_arcmin(dt_days)

    cropped, mask = crop_to_polygon(image_data, wcs, polygon_ra, polygon_dec)
    if mask.sum() < min_mask_pixels:
        return []

    edges = simple_edge_map(cropped, threshold_sigma=2.5, ignore_mask=mask)

    r_min_px = max(1, int(arcmin_to_pixels(theta_min)))
    r_max_px = int(arcmin_to_pixels(theta_max)) + 1
    radii_px = range(r_min_px, r_max_px, radius_step_px)

    acc, candidates = circular_hough_transform(edges, radii_px, vote_threshold_frac=vote_threshold_frac)
    candidates.sort(key=lambda c: -c[3])  # highest votes first

    results = []
    for y, x, r_px, votes in candidates:
        ra_c, dec_c = wcs.wcs_pix2world(x, y, 0)
        radius_arcmin = (r_px * ARCSEC_PER_PIXEL) / 60.0
        results.append({
            "x_px": x, "y_px": y, "ra": float(ra_c), "dec": float(dec_c),
            "radius_arcmin": radius_arcmin, "votes": votes,
            "predicted_theta_arcmin": theta_center,
        })
    return results

In [ ]:

# %% [markdown]
# ## Step 3: Non-max suppression - collapse duplicate ring candidates

# %%
"""
Non-max suppression (NMS) for circular Hough transform candidates.

Problem: a single real ring produces many near-duplicate high-vote
candidates (center off by a pixel, radius off by a pixel, etc.) -
not because there are many rings, but because the accumulator is
forgiving. NMS collapses each such cluster down to one entry: the
highest-voted candidate in that cluster.

Algorithm (greedy, the standard approach):
  1. Sort all candidates by votes, descending.
  2. Take the top one - it's a confirmed detection.
  3. Remove every remaining candidate close enough in (center, radius)
     to be "the same ring" as that detection.
  4. Repeat with whatever's left until none remain.
"""

import numpy as np


def non_max_suppress_circles(candidates, center_dist_px=10, radius_dist_px=5):
    """
    candidates: list of dicts, each with at least 'x_px', 'y_px',
                and a radius field (pixels) plus 'votes'.
                Works with the dicts produced by run_halo_search,
                which use 'radius_arcmin' instead of a pixel radius -
                pass radius_px explicitly via radius_key if needed.
    center_dist_px: two candidates are "the same ring" if their
                centers are within this many pixels of each other.
    radius_dist_px: ...and their radii are within this many pixels
                of each other. (If your radius field is in arcmin,
                convert this threshold to arcmin before calling, or
                use the wrapper below.)

    Returns a new list, one entry per distinct ring, sorted by votes
    descending - the highest-voted member of each cluster.
    """
    remaining = sorted(candidates, key=lambda c: -c["votes"])
    kept = []

    while remaining:
        best = remaining.pop(0)
        kept.append(best)

        survivors = []
        for c in remaining:
            d_center = np.hypot(c["x_px"] - best["x_px"], c["y_px"] - best["y_px"])
            d_radius = abs(c["_radius_px"] - best["_radius_px"])
            if d_center > center_dist_px or d_radius > radius_dist_px:
                survivors.append(c)
            # else: close enough to `best` -> same ring, drop it
        remaining = survivors

    return kept


def non_max_suppress_arcmin_results(results, arcsec_per_pixel=4.0,
                                     center_dist_arcmin=2.0, radius_dist_arcmin=1.0):
    """
    Convenience wrapper for the dicts produced by run_halo_search
    (step2_halo_search.py), which carry 'radius_arcmin' rather than
    a pixel radius. Converts thresholds to pixels internally, then
    calls non_max_suppress_circles.

    IMPORTANT - these defaults are a starting guess, not a verified
    value: how far a single real ring's raw candidates spread out in
    (center, radius) depends on your edge detector's noise level and
    the angle/radius step sizes used in the CHT pass. Tune both
    numbers against your own data: take one obvious/known ring's raw
    (pre-NMS) candidates, look at how spread out they are, and set
    these thresholds comfortably larger than that spread. Too tight
    -> one real ring gets reported as many "distinct" detections.
    Too loose -> two genuinely separate nearby rings get merged into
    one.
    """
    px_per_arcmin = 60.0 / arcsec_per_pixel

    # stash a pixel-radius field for the suppressor to compare on
    annotated = []
    for r in results:
        r = dict(r)  # don't mutate caller's dicts
        r["_radius_px"] = r["radius_arcmin"] * px_per_arcmin
        annotated.append(r)

    kept = non_max_suppress_circles(
        annotated,
        center_dist_px=center_dist_arcmin * px_per_arcmin,
        radius_dist_px=radius_dist_arcmin * px_per_arcmin,
    )

    for r in kept:
        del r["_radius_px"]
    return kept

In [ ]:

# %% [markdown]
# ## Step 4: Download skytile FITS files into skytiles/{event}/

# %%
"""
Downloads the eROSITA DR1 image FITS files for every skytile that
overlaps each GW event's boundary polygon, saving into:

    skytiles/{event_name}/{filename}.fits.gz

Per-event tile lookup ignores any single "center" coordinate (GW
localization regions aren't well represented by one center+radius -
this was already the approach in step1_tile_grid.py's
find_all_skytiles_for_event, which samples points across the whole
polygon including disjoint fragments).

Filename construction follows the documented scheme:
    https://erosita.mpe.mpg.de/dr1/eSASS4DR1/eSASS4DR1_ProductsDescription/file_naming_scheme_dr1.html
    PQii_jjjjjj_klm_nnnnnn_Rooo.fits

CONFIRMED against a real directory listing (srvmap 174079, i.e.
RA-tile 174 / Dec-tile 079):
  - Count IMAGE files live in EXP_010, not DET_010. DET_010 only has
    AreaTab/SensMap/ApeSensMap/BackgrImage/ExposureMap - no Image.
  - The jjjjjj field in the FILENAME is written DDDRRR (Dec-tile then
    RA-tile) - the REVERSE of the directory path order, which is
    RRR/DDD (matches the directory example /174/079/). Example seen:
      directory: /174/079/EXP_010/
      filename:  em01_079174_024_Image_c010.fits.gz
                          ^^^^^^ = DDD(079) + RRR(174)
  - Files are gzip-compressed (.fits.gz), not plain .fits.
    astropy.io.fits.open() handles gzip transparently, so this only
    matters for the saved filename / requests.get() URL, not for
    reading the file afterward.
  - klm=024 is band index 4 (0.2-2.3 keV, eROSITA's standard
    soft+medium combined band) with k=0 (all cameras), l=2 (filter
    wheel: filter). klm=021..027 cover bands 1-7; there is no 020
    Image (020 only has an EventList in this listing).
"""

import os
import re
import csv
import requests
from collections import defaultdict


ARCHIVE_BASE = "https://erosita.mpe.mpg.de/dr1/erodat/data/download"

# klm=024 -> all cameras, filter wheel, 0.2-2.3 keV band (confirmed to
# exist as em01_{DDDRRR}_024_Image_c010.fits.gz in the EXP_010 listing)
GUESSED_FILENAME_TEMPLATE = "em01_{ddd:03d}{rrr:03d}_024_Image_c010.fits.gz"


def tile_directory_url(srvmap, product="EXP", proc="010"):
    """
    Directory path is /{RRR}/{DDD}/{PRODUCT}_{proc}/ - confirmed
    against the real example /174/079/EXP_010/, where srvmap=174079
    (i.e. srvmap as returned by the skytile search API is RRRDDD).
    """
    srvmap_str = f"{srvmap:06d}"
    rrr, ddd = srvmap_str[:3], srvmap_str[3:]
    return f"{ARCHIVE_BASE}/{rrr}/{ddd}/{product}_{proc}/", rrr, ddd


def guessed_filename(rrr, ddd):
    """
    NOTE the swap: the directory is RRR/DDD, but the filename's
    jjjjjj field is written DDD+RRR (reversed) - see module docstring.
    """
    return GUESSED_FILENAME_TEMPLATE.format(ddd=int(ddd), rrr=int(rrr))


def find_image_filename_from_listing(directory_url):
    """
    Fallback: fetch the directory's HTML listing and regex out a
    filename that looks like an Image product. Used only if the
    guessed filename 404s (e.g. if a tile only has some energy bands
    processed, or the band/camera digits differ from our default).
    """
    resp = requests.get(directory_url, timeout=30)
    resp.raise_for_status()
    candidates = re.findall(r'href="([^"]*_024_Image[^"]*\.fits\.gz)"', resp.text)
    if not candidates:
        # widen: any Image file, any band, in case band 024 isn't present for this tile
        candidates = re.findall(r'href="([^"]*Image[^"]*\.fits\.gz)"', resp.text)
    return candidates[0] if candidates else None


def download_skytile_image(srvmap, dest_path, product="EXP", proc="010"):
    """
    Tries the guessed filename first (one request, fast path).
    Falls back to scraping the directory listing if that 404s.
    Returns True on success, False if nothing could be downloaded.
    """
    directory_url, rrr, ddd = tile_directory_url(srvmap, product=product, proc=proc)
    guessed_name = guessed_filename(rrr, ddd)
    guessed_url = directory_url + guessed_name

    resp = requests.get(guessed_url, timeout=120)
    if resp.status_code == 200:
        with open(dest_path, "wb") as f:
            f.write(resp.content)
        return True

    # fallback: scrape listing for the real filename
    real_name = find_image_filename_from_listing(directory_url)
    if real_name is None:
        print(f"    [fail] srvmap={srvmap}: no Image file found in {directory_url}")
        return False

    real_url = directory_url + real_name
    resp = requests.get(real_url, timeout=120)
    if resp.status_code != 200:
        print(f"    [fail] srvmap={srvmap}: {real_url} -> HTTP {resp.status_code}")
        return False

    with open(dest_path, "wb") as f:
        f.write(resp.content)
    return True


def load_events_csv(csv_path):
    events = defaultdict(list)
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            events[row["event_name"]].append(row)
    return events


def download_all_skytiles(csv_path, boundary_dir, out_root="skytiles",
                           product="EXP", proc="010"):
    """
    For every unique event in csv_path:
      - load its _boundary.json from boundary_dir
      - find every skytile overlapping any part of the boundary
        (all blobs, not just the main one - and ignoring any single
        center coordinate, per your instruction)
      - download each tile's image FITS into out_root/{event_name}/
    """
    events = load_events_csv(csv_path)

    for event_name in events:
        event_dir = os.path.join(out_root, event_name)
        os.makedirs(event_dir, exist_ok=True)

        boundary_path = os.path.join(boundary_dir, f"{event_name}_boundary.json")
        tiles, samples, blobs = find_all_skytiles_for_event(boundary_path)

        print(f"[{event_name}] {len(tiles)} skytile(s) overlap the boundary region")

        for tile in tiles:
            srvmap = tile["srvmap"]
            dest_path = os.path.join(event_dir, f"srvmap_{srvmap:06d}.fits.gz")

            if os.path.exists(dest_path):
                print(f"    [skip] srvmap={srvmap}: already downloaded")
                continue

            ok = download_skytile_image(srvmap, dest_path, product=product, proc=proc)
            status = "ok" if ok else "FAILED"
            print(f"    [{status}] srvmap={srvmap} -> {dest_path}")


In [ ]:


# %% [markdown]
# ## Step 5: Orchestrator - one CHT search per GW event

# %%
"""
Top-level pipeline: exactly ONE circular Hough transform search per
unique GW event, restricted to the sky region defined by that
event's _boundary.json polygon(s).

For each event:
  1. Average dt_days across its CSV rows -> one predicted theta(dt)
     radius window (their spread is tiny - see note below - so this
     loses essentially no information).
  2. Find every eROSITA skytile whose footprint intersects ANY blob
     of the boundary polygon (find_all_skytiles_for_event).
  3. For each such tile: download FITS, crop to the polygon, run CHT
     over the theta(dt) radius window only.
  4. Merge candidates from all tiles belonging to this event into one
     result list - this *is* the "one search per event", just spread
     across however many tiles the polygon happens to cover.

Note on averaging dt_days: within a single event the 7 observation
rows differ by under a day out of 100-280+ total days, so the
predicted theta shifts by a few percent at most - far smaller than
the +-20% RADIUS_UNCERTAINTY_FRAC band already applied. Averaging is
safe here; it would NOT be safe if dt varied by, e.g., 50% within an
event.
"""

import csv
from collections import defaultdict

from astropy.wcs import WCS


# (load_events_csv already defined above in Step 4)


def mean_dt_days_for_event(event_rows):
    dts = [parse_relative_time_to_days(r["time_relative_to_gw_detection"]) for r in event_rows]
    return sum(dts) / len(dts)


def boundary_path_for_event(event_name, boundary_dir=BOUNDARY_DIR):
    return f"{boundary_dir}/{event_name}_boundary.json"


def _bbox_could_overlap(tile_ra_c, tile_dec_c, tile_half_width_deg, poly_ra, poly_dec):
    """
    Cheap rectangle-vs-rectangle overlap check (not a true polygon
    intersection - just enough to quickly rule out the vast majority
    of (tile, blob) pairs that obviously can't overlap, e.g. a tile
    near blob 0 being tested against a tiny fragment blob clear on
    the other side of the sky). Saves a full crop_to_polygon + CHT
    call on pairs that have no chance of overlapping.

    A small amount of slack is already baked into tile_half_width_deg
    by the caller, so this errs on the side of "maybe overlap -> run
    the real check" rather than risking a false negative.
    """
    poly_ra_min, poly_ra_max = min(poly_ra), max(poly_ra)
    poly_dec_min, poly_dec_max = min(poly_dec), max(poly_dec)

    tile_ra_min = tile_ra_c - tile_half_width_deg
    tile_ra_max = tile_ra_c + tile_half_width_deg
    tile_dec_min = tile_dec_c - tile_half_width_deg
    tile_dec_max = tile_dec_c + tile_half_width_deg

    ra_overlap = tile_ra_min <= poly_ra_max and tile_ra_max >= poly_ra_min
    dec_overlap = tile_dec_min <= poly_dec_max and tile_dec_max >= poly_dec_min
    return ra_overlap and dec_overlap


def run_one_search_per_event(csv_path, boundary_dir=BOUNDARY_DIR, product="EXP", proc="010"):
    """
    Returns {event_name: {"theta_window": (...), "tiles": [...],
                            "candidates": [...]}}
    """
    events = load_events_csv(csv_path)
    all_results = {}

    for event_name, rows in events.items():
        dt_days = mean_dt_days_for_event(rows)
        theta_min, theta_max, theta_center = predicted_radius_window_arcmin(dt_days)

        boundary_path = boundary_path_for_event(event_name, boundary_dir)
        tiles, samples, blobs = find_all_skytiles_for_event(boundary_path)

        print(f"[{event_name}] dt={dt_days:.1f}d  theta_window=[{theta_min:.2f},{theta_max:.2f}] arcmin  "
              f"-> {len(tiles)} skytile(s) cover the boundary region")

        # combine all blob vertices into one polygon list per tile crop
        # (crop_to_polygon's point-in-polygon test is run per-blob and
        # OR'd together, since a tile may only overlap one fragment)
        event_candidates = []
        for tile in tiles:
            srvmap = tile["srvmap"]
            try:
                url = build_download_url(srvmap, product=product, proc=proc)
                hdul = download_fits(url)
                image = hdul[0].data
                wcs = WCS(hdul[0].header)
            except Exception as e:
                print(f"    [skip] srvmap={srvmap}: download/parse failed ({e})")
                continue

            # search each blob's polygon against this tile separately
            # (a tile might overlap blob 0 and blob 3, say, but not blob 1).
            # Cheap bounding-box pre-check first: if the tile's center
            # (+ ~half its 3.6deg width) can't possibly reach the
            # blob's bounding box, skip the expensive crop+CHT call.
            tile_ra_c, tile_dec_c = tile["ra_cen"], tile["de_cen"]
            tile_half_width_deg = 1.9  # 3.6deg tile, plus a little slack

            for blob_id, points in blobs.items():
                if len(points) < 3:
                    continue  # can't crop to a 1-2 point "polygon"
                poly_ra = [p[0] for p in points]
                poly_dec = [p[1] for p in points]

                if not _bbox_could_overlap(tile_ra_c, tile_dec_c, tile_half_width_deg,
                                            poly_ra, poly_dec):
                    continue

                results = run_halo_search(image, wcs, poly_ra, poly_dec, dt_days)
                for r in results:
                    r["srvmap"] = srvmap
                    r["blob_id"] = blob_id
                event_candidates.extend(results)

        event_candidates.sort(key=lambda c: -c["votes"])
        n_raw = len(event_candidates)

        # Collapse the cloud of near-duplicate (x,y,r) candidates that
        # a single real ring produces down to one entry per distinct
        # ring (see nms_circles.py for why this is needed).
        suppressed_candidates = non_max_suppress_arcmin_results(event_candidates)

        all_results[event_name] = {
            "dt_days": dt_days,
            "theta_window_arcmin": (theta_min, theta_max, theta_center),
            "tiles": tiles,
            "candidates": suppressed_candidates,
            "n_raw_candidates": n_raw,
        }

    return all_results
